# Pemodelan Struktural (Dimension Modeling)


## Inisialisasi Spark Session dan Setup Path


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Inisialisasi SparkSession dengan konfigurasi dynamic partition overwrite
spark = (
    SparkSession.builder
    .appName('Retail_Dimensional_Modeling')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic') # Kunci idempotensi untuk overwrite partisi secara dinamis
    .getOrCreate()
)

raw_path = '../data/raw/olist'
silver_path = '../data/silver'

print("SparkSession initialized successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/17 12:45:13 WARN Utils: Your hostname, lathief-laptop, resolves to a loopback address: 127.0.1.1; using 192.168.1.9 instead (on interface wlp0s20f3)
26/09/17 12:45:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lathief/coding/myproject/project4-aws/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/17 12:45:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession initialized successfully.


## 2. Membangun Dimensi Kalender (dim_date)


In [2]:
df_date_range = spark.sql(
    """
    select
        explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as calendar_date
    """
)

dim_date = (
    df_date_range
    .withColumn('date_key', F.date_format('calendar_date', 'yyyyMMdd').cast(IntegerType()))
    .withColumn('year', F.year('calendar_date'))
    .withColumn('month', F.month('calendar_date'))
    .withColumn('month_name', F.date_format('calendar_date', 'MMMM'))
    .withColumn('day', F.dayofmonth('calendar_date'))
    .withColumn('day_of_week', F.dayofweek('calendar_date'))
    .withColumn('day_name', F.date_format('calendar_date', 'EEEE'))
    .withColumn('quarter', F.quarter('calendar_date'))
    .withColumn('is_weekend', F.when(F.col('day_of_week').isin(1,7), True).otherwise(False))
)

dim_date.printSchema()
dim_date.show(5, truncate=False)

root
 |-- calendar_date: date (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- year: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- day: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- day_name: string (nullable = false)
 |-- quarter: integer (nullable = false)
 |-- is_weekend: boolean (nullable = false)

+-------------+--------+----+-----+----------+---+-----------+--------+-------+----------+
|calendar_date|date_key|year|month|month_name|day|day_of_week|day_name|quarter|is_weekend|
+-------------+--------+----+-----+----------+---+-----------+--------+-------+----------+
|2016-01-01   |20160101|2016|1    |January   |1  |6          |Friday  |1      |false     |
|2016-01-02   |20160102|2016|1    |January   |2  |7          |Saturday|1      |true      |
|2016-01-03   |20160103|2016|1    |January   |3  |1          |Sunday  |1      |true      |
|2016-01-04   |20160104|2016|1    |

In [3]:
# Tulis ke silver layer (Format parquet, overwrite mode)
dim_date.write.mode('overwrite').parquet(f'{silver_path}/dim_date')
print('dim_date successfully saved to silver layer.')

dim_date successfully saved to silver layer.


## 3. Membangun Dimensi Produk (dim_product) - scd-type 1


In [4]:
df_products_raw = spark.read.csv(f'{raw_path}/olist_products_dataset.csv', header=True, inferSchema=True)
df_categories_raw = spark.read.csv(f'{raw_path}/product_category_name_translation.csv', header=True, inferSchema=True)

# 1. Enrichment translasi kategori & sanitasi missing values
dim_product = (
    df_products_raw
    .join(df_categories_raw, on='product_category_name', how='left')
    .select(
        F.col('product_id'),
        # Imputasi: jika null, beri label 'Unknown' agar join downstream tidak pecah
        F.coalesce(F.col('product_category_name_english'), F.lit('unknown')).alias('category_name'),
        F.coalesce(F.col('product_weight_g'), F.lit(0)).alias('weight_g'),
        F.coalesce(F.col('product_length_cm'), F.lit(0)).alias('length_cm'),
        F.coalesce(F.col('product_height_cm'), F.lit(0)).alias('height_cm'),
        F.coalesce(F.col('product_width_cm'), F.lit(0)).alias('width_cm'),
        # Timestamp metadata audit (kapan record ini diproses ke Silver)
        F.current_timestamp().alias('updated_at')
    )
    .dropDuplicates(['product_id'])  # Menjamin uniqueness Axiom: product_id adalah primary key
)

dim_product.printSchema()
dim_product.show(5, truncate=False)

root
 |-- product_id: string (nullable = true)
 |-- category_name: string (nullable = false)
 |-- weight_g: integer (nullable = false)
 |-- length_cm: integer (nullable = false)
 |-- height_cm: integer (nullable = false)
 |-- width_cm: integer (nullable = false)
 |-- updated_at: timestamp (nullable = false)



+--------------------------------+--------------+--------+---------+---------+--------+--------------------------+
|product_id                      |category_name |weight_g|length_cm|height_cm|width_cm|updated_at                |
+--------------------------------+--------------+--------+---------+---------+--------+--------------------------+
|00066f42aeeb9f3007548bb9d3f33c38|perfumery     |300     |20       |16       |16      |2026-09-17 05:45:27.473509|
|00088930e925c41fd95ebfe695fd2655|auto          |1225    |55       |10       |26      |2026-09-17 05:45:27.473509|
|0011c512eb256aa0dbbb544d8dffcf6e|auto          |100     |16       |15       |16      |2026-09-17 05:45:27.473509|
|00126f27c813603687e6ce486d909d01|cool_stuff    |700     |25       |5        |15      |2026-09-17 05:45:27.473509|
|001795ec6f1b187d37335e1c4704762e|consoles_games|600     |30       |20       |20      |2026-09-17 05:45:27.473509|
+--------------------------------+--------------+--------+---------+---------+--

In [5]:
dim_product.write.mode('overwrite').parquet(f'{silver_path}/dim_product')
print('dim_product successfully saved to silver layer.')

dim_product successfully saved to silver layer.


## 4. Membangun dim_customer scd-type2


In [6]:
from pyspark.sql.window import Window

df_customers_raw = spark.read.csv(f'{raw_path}/olist_customers_dataset.csv', header=True, inferSchema=True)
df_orders_raw = spark.read.csv(f'{raw_path}/olist_orders_dataset.csv', header=True, inferSchema=True)

# 1. Hubungkan customer dengan purchase timestamp dari order untuk tahu kapan profil ini aktif
df_cust_orders = (
    df_customers_raw
    .join(df_orders_raw.select('order_id', 'customer_id', 'order_purchase_timestamp'), on='customer_id', how='inner')
    .withColumn('purchase_date', F.to_date(F.to_timestamp('order_purchase_timestamp')))
    .select(
        'customer_unique_id',
        F.col('customer_zip_code_prefix').alias('zip_code'),
        F.col('customer_city').alias('city'),
        F.col('customer_state').alias('state'),
        'purchase_date'
    )
)

# 2. Ambil tanggal pertama kali kombinasi (customer + lokasi) muncul (effective start date)
df_cust_history = (
    df_cust_orders
    .groupBy('customer_unique_id', 'zip_code', 'city', 'state')
    .agg(F.min('purchase_date').alias('start_date'))
)

# 3. Urutkan riwayat per customer_unique_id menggunakan window function untuk menentukan end_date & is_current
window_spec = Window.partitionBy('customer_unique_id').orderBy('start_date')

dim_customer_scd2 = (
    df_cust_history
    # end_date adalah start_date dari versi berikutnya (lead), atau '9999-12-31' jika versi terbaru
    .withColumn('next_start_date', F.lead('start_date').over(window_spec))
    .withColumn('end_date', F.coalesce(F.date_sub(F.col('next_start_date'), 1), F.to_date(F.lit('9999-12-31'))))  # Jika tidak ada next_start_date, berarti ini adalah versi terakhir
    # is_current bernilai True hanya jika record tidak memiliki versi penerus (next_start_date is null)
    .withColumn('is_current', F.col('next_start_date').isNull())
    # Generate deterministic surrogate key untuk dim_customer_scd2 pakai MD5 hash
    .withColumn('customer_sk', F.md5(F.concat_ws('||', 'customer_unique_id', 'city', 'state', 'start_date')))
    .select(
        'customer_sk',
        'customer_unique_id',
        'zip_code',
        'city',
        'state',
        'start_date',
        'end_date',
        'is_current'
    )
)


dim_customer_scd2.printSchema()
dim_customer_scd2.show(5, truncate=False)

root
 |-- customer_sk: string (nullable = false)
 |-- customer_unique_id: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = false)

+--------------------------------+--------------------------------+--------+--------------+-----+----------+----------+----------+
|customer_sk                     |customer_unique_id              |zip_code|city          |state|start_date|end_date  |is_current|
+--------------------------------+--------------------------------+--------+--------------+-----+----------+----------+----------+
|976363f92dd06b4f94d6566de7b3be4f|0006fdc98a402fceb4eb0ee528f6a8d4|29400   |mimoso do sul |ES   |2017-07-18|9999-12-31|true      |
|a80f9b868fe0d7a5ce127700f49b3037|00090324bbad0e9342388303bb71ba0a|13054   |campinas      |SP   |2018-03-24|9999-12-31|true      |
|6eb24d81

## 5. Verifikasi Riwayat Customer yang Pindah Lokasi


In [7]:
# Ambil contoh customer yang di-EDA terbukti pindah kota
sample_customer = 'd44ccec15f5f86d14d6a2cfa67da1975'

print(f'Riwayat SCD2 untuk customer: {sample_customer}')
dim_customer_scd2.filter(F.col('customer_unique_id') == sample_customer).show(truncate=False)

# Simpan ke silver layer
dim_customer_scd2.write.mode('overwrite').parquet(f'{silver_path}/dim_customer')
print('dim_customer (SCD Type 2) successfully saved to silver layer.')

Riwayat SCD2 untuk customer: d44ccec15f5f86d14d6a2cfa67da1975
+--------------------------------+--------------------------------+--------+----------+-----+----------+----------+----------+
|customer_sk                     |customer_unique_id              |zip_code|city      |state|start_date|end_date  |is_current|
+--------------------------------+--------------------------------+--------+----------+-----+----------+----------+----------+
|b56262969fb77822453670af84965964|d44ccec15f5f86d14d6a2cfa67da1975|3533    |sao paulo |SP   |2017-05-30|2017-09-12|false     |
|9731c4c7c012add288fc23e284402141|d44ccec15f5f86d14d6a2cfa67da1975|88371   |navegantes|SC   |2017-09-13|2017-11-09|false     |
|eb9476e48511f61c8392d4bd547b5fc0|d44ccec15f5f86d14d6a2cfa67da1975|62800   |aracati   |CE   |2017-11-10|9999-12-31|true      |
+--------------------------------+--------------------------------+--------+----------+-----+----------+----------+----------+



dim_customer (SCD Type 2) successfully saved to silver layer.


## 6. Membangun fact_orders dengan Point-in-Time SCD2 Lookup


In [9]:
df_items_raw = spark.read.csv(f'{raw_path}/olist_order_items_dataset.csv', header=True, inferSchema=True)
df_order_raw = spark.read.csv(f'{raw_path}/olist_orders_dataset.csv', header=True, inferSchema=True)
df_customers_raw = spark.read.csv(f'{raw_path}/olist_customers_dataset.csv', header=True, inferSchema=True)

# Baca dim_customer yang sudah tersimpan di Silver
dim_customer = spark.read.parquet(f'{silver_path}/dim_customer')

# 1. Siapkan transaksi dasar (orders + items + customer_unique_id)
df_transaksi = (
    df_items_raw
    .join(df_orders_raw, on='order_id', how='inner')
    .join(df_customers_raw.select('customer_id', 'customer_unique_id'), on='customer_id', how='inner')
    .withColumn('purchase_ts', F.to_timestamp('order_purchase_timestamp'))
    .withColumn('purchase_date', F.to_date('purchase_ts'))
    .withColumn('date_key', F.date_format('purchase_date', 'yyyyMMdd').cast(IntegerType()))
)

# 2. Poin-in-Time Join ke dim_customer (Range Join: purchase_date BETWEEN start_date AND end_date)
fact_orders = (
    df_transaksi
    .join(dim_customer, on=(
        (df_transaksi.customer_unique_id == dim_customer.customer_unique_id)
        & (df_transaksi.purchase_date >= dim_customer.start_date)
        & (df_transaksi.purchase_date <= dim_customer.end_date)
    ), how='inner')
    .select(
        # Degenerate Dimension / Business Key
        df_transaksi['order_id'],
        df_transaksi['order_item_id'],

        # Foreign Key ke Dimensi
        df_transaksi['date_key'],
        df_transaksi['product_id'],
        dim_customer['customer_sk'],

        # Transaction Status
        df_transaksi['order_status'],

        # Additive Fact Measures (Nilai Numerik yang Sah di-SUM)
        df_transaksi['price'].alias('item_price'),
        df_transaksi['freight_value'],
        (df_transaksi['price'] + df_transaksi['freight_value']).alias('total_item_value'),

        # Kolom Partisi Waktu (Untuk Storage & Idempotensi)
        df_transaksi['purchase_date']
    )
)

print(f'Total baris Fact Orders: {fact_orders.count()}')
fact_orders.printSchema()
fact_orders.show(5, truncate=False)

Total baris Fact Orders: 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- item_price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- purchase_date: date (nullable = true)

+--------------------------------+-------------+--------+--------------------------------+--------------------------------+------------+----------+-------------+----------------+-------------+
|order_id                        |order_item_id|date_key|product_id                      |customer_sk                     |order_status|item_price|freight_value|total_item_value|purchase_date|
+--------------------------------+-------------+--------+--------------------------------+--------------------------------+------------+------

## 7. Menulis ke Silver dengan Partisi Harian


In [10]:
# Tulis ke Silver Layer dengan partisi harian
(
    fact_orders
    .write
    .mode('overwrite')
    .partitionBy('purchase_date')
    .parquet(f'{silver_path}/fact_orders')
)

print('fact_orders successfully saved to silver layer with daily partitioning.')

fact_orders successfully saved to silver layer with daily partitioning.


## 8. The Idempotency Test


In [12]:
# Pilih satu tanggal spesifik untuk uji rerun
test_date = '2017-10-02'

# 1. Hitung total baris awal di tanggal tersebut
initial_count = spark.read.parquet(f'{silver_path}/fact_orders').filter(F.col('purchase_date') == test_date).count()
print(f'Jumlah baris awal untuk {test_date}: {initial_count}')

# 2. Simulasikan Airflow mmenjalankan ulang batch untuk tanggal 2017-10-02 saja
single_day_batch = fact_orders.filter(F.col('purchase_date') == test_date)

# Tulis ulang dengan mode overwrite (karena spark.sql.sources.partitionOverWriteMode = dynamic, Hanya partisi 2017-10-02 yang disentuh)
(
    single_day_batch
    .write
    .mode('overwrite')
    .partitionBy('purchase_date')
    .parquet(f'{silver_path}/fact_orders')
)

# 3. hitung ulang total baris setelah rerun
post_rerun_count = spark.read.parquet(f'{silver_path}/fact_orders').filter(F.col('purchase_date') == test_date).count()
print(f'Jumlah baris setelah rerun untuk {test_date}: {post_rerun_count}')

# 4. Assert Idempotensi: initial_count HARUS SAMA DENGAN post_rerun_count
assert initial_count == post_rerun_count, 'Gagal: Terjadi duplikasi data!'
print('Idempotency Assertion Passed: Tidak ada duplikasi data saat pipeline dijalankan ulang.')


Jumlah baris awal untuk 2017-10-02: 166


Jumlah baris setelah rerun untuk 2017-10-02: 166
Idempotency Assertion Passed: Tidak ada duplikasi data saat pipeline dijalankan ulang.
